In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .getOrCreate()

Найти велосипед с максимальным временем пробега.

Найти наибольшее геодезическое расстояние между станциями.

Найти путь велосипеда с максимальным временем пробега через станции.

Найти количество велосипедов в системе.

Найти пользователей потративших на поездки более 3 часов.

In [ ]:
# 1. Устанавливаем библиотеку
!pip install kagglehub

import kagglehub

# 2. Скачиваем датасет
path = kagglehub.dataset_download("benhamner/sf-bay-area-bike-share")

print("Путь к файлам:", path)

100%|██████████| 554M/554M [00:08<00:00, 71.6MB/s]

Extracting files...


Путь к файлам: /root/.cache/kagglehub/datasets/benhamner/sf-bay-area-bike-share/versions/2


In [ ]:
import os

# Путь, который выдал kagglehub
dataset_path = "/root/.cache/kagglehub/datasets/benhamner/sf-bay-area-bike-share/versions/2"

# Формируем полные пути к конкретным файлам
trips_path = os.path.join(dataset_path, "trip.csv")
stations_path = os.path.join(dataset_path, "station.csv")


trips = spark.read.csv(trips_path, header=True, inferSchema=True)
stations = spark.read.csv(stations_path, header=True, inferSchema=True)

print("Данные успешно прочитаны!")

Данные успешно прочитаны!


In [ ]:
# --- ЗАДАЧА 1: Велосипед с максимальным временем пробега ---
bike_max = trips.groupBy("bike_id").agg(F.sum("duration").alias("total_duration")) \
                .orderBy(F.desc("total_duration")).first()

print(f"1. Велосипед ID {bike_max['bike_id']} проехал больше всех: {bike_max['total_duration']} секунд.")

1. Велосипед ID 535 проехал больше всех: 18611693 секунд.


In [ ]:
# --- ЗАДАЧА 2: Наибольшее геодезическое расстояние между станциями ---
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    # Формула для расчета расстояния на сфере
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

# Собираем координаты всех станций в список
st_list = stations.select("long", "lat").collect()
distances = [haversine(st_list[i][0], st_list[i][1], st_list[j][0], st_list[j][1])
             for i in range(len(st_list)) for j in range(i + 1, len(st_list))]

print(f"2. Максимальное геодезическое расстояние: {max(distances):.2f} км.")

2. Максимальное геодезическое расстояние: 69.92 км.


In [ ]:
# --- ЗАДАЧА 3: Путь велосипеда с макс. временем пробега через станции ---
# Берем ID из первой задачи
top_bike_id = bike_max['bike_id']
bike_path = trips.filter(F.col("bike_id") == top_bike_id).orderBy("start_date")

print(f"3. Путь велосипеда {top_bike_id} (первые 5 станций):")
bike_path.select("start_station_name", "end_station_name", "start_date").show(5, truncate=False)

3. Путь велосипеда 535 (первые 5 станций):
+-----------------------------------+----------------------------------------+---------------+
|start_station_name                 |end_station_name                        |start_date     |
+-----------------------------------+----------------------------------------+---------------+
|Mechanics Plaza (Market at Battery)|Embarcadero at Sansome                  |1/1/2014 13:42 |
|Embarcadero at Sansome             |Market at 4th                           |1/1/2014 18:51 |
|Market at 4th                      |South Van Ness at Market                |1/1/2014 19:48 |
|Market at 10th                     |Powell Street BART                      |1/10/2014 20:13|
|Embarcadero at Folsom              |San Francisco Caltrain (Townsend at 4th)|1/10/2014 8:09 |
+-----------------------------------+----------------------------------------+---------------+
only showing top 5 rows


In [ ]:
# --- ЗАДАЧА 4: Количество велосипедов в системе ---
total_bikes = trips.select("bike_id").distinct().count()
print(f"4. Всего уникальных велосипедов: {total_bikes}")

4. Всего уникальных велосипедов: 700


In [ ]:
# --- ЗАДАЧА 5: Пользователи, потратившие более 3 часов (10800 сек) ---
users_3h = trips.groupBy("zip_code").agg(F.sum("duration").alias("total_time")) \
                .filter(F.col("total_time") > 10800)

print("5. ТОП 5 пользователей, накатавших больше 3 часов:")
users_3h.orderBy(F.desc("total_time")).show(5)

5. Пользователи (zip_code), накатавшие больше 3 часов (топ-5):
+--------+----------+
|zip_code|total_time|
+--------+----------+
|   94107|  49757162|
|     nil|  45725550|
|    NULL|  27723273|
|   94105|  25596128|
|   94133|  21637675|
+--------+----------+
only showing top 5 rows
